# 未然 ForeSure: AMD ROCm GPU 1,000,000 次蒙地卡羅巨災精算壓力測試
### FUTUREMODE x SITCON BUILDMODE 2026 · 國泰金控 AI AGENT 賽道
本筆記本於 **AMD AUP Learning Cloud (tpe.aupcloud.io)** 之 **Deep Learning Course** 環境執行。
利用 AMD ROCm GPU 張量核心加速複合泊松-極端值巨災模型（Compound Poisson-Pareto Process），
於秒級內完成 1,000,000 次巨災情境採樣，計算 Solvency II / TW-ICS 規範之 99.5% VaR、TVaR 與清償資本適足加成。

In [ ]:
# 1. 驗證 AMD ROCm GPU 硬體加速就緒狀態
import torch
import sys
import time

print(f"Python Version: {sys.version}")
print(f"PyTorch Version: {torch.__version__}")
is_rocm = torch.cuda.is_available()
print(f"ROCm GPU Available: {is_rocm}")
if is_rocm:
    print(f"Device Name: {torch.cuda.get_device_name(0)}")
    print(f"Device Count: {torch.cuda.device_count()}")
    device = torch.device("cuda:0")
else:
    print("Warning: GPU not detected, falling back to CPU vectorized execution.")
    device = torch.device("cpu")

In [ ]:
# 2. 定義 GPU 向量化百萬次巨災蒙地卡羅模擬核心
import math
import json

def simulate_catastrophe_rocm(peril_name, annual_frequency, mean_loss_usd, iterations=1_000_000, seed=42):
    torch.manual_seed(seed)
    start_t = time.perf_counter()
    
    # 參數設定：厚尾波動係數 (Heavy-tail parameter)
    sigma = 0.95 if peril_name in ["typhoon", "earthquake", "climate"] else 0.75
    mu = math.log(max(1.0, mean_loss_usd)) - 0.5 * (sigma ** 2)
    
    # 1. 向量化生成 1,000,000 年的事件發生頻率 N ~ Poisson(lambda)
    lam = max(0.001, float(annual_frequency))
    rates = torch.full((iterations,), lam, device=device, dtype=torch.float32)
    event_counts = torch.poisson(rates)
    
    # 2. 向量化累積年度巨災損失 S = sum(X_j)
    annual_losses = torch.zeros(iterations, device=device, dtype=torch.float32)
    max_events = int(torch.max(event_counts).item())
    
    for k in range(1, min(max_events + 1, 15)):
        mask = event_counts >= k
        n_active = int(mask.sum().item())
        if n_active > 0:
            norm_sample = torch.randn(n_active, device=device, dtype=torch.float32)
            loss_sample = torch.exp(mu + sigma * norm_sample)
            annual_losses[mask] += loss_sample
            
    # 3. 排序以計算 Solvency II / TW-ICS 99.5% 尾端損失 (200年一遇巨災)
    sorted_losses, _ = torch.sort(annual_losses)
    elapsed_ms = (time.perf_counter() - start_t) * 1000
    
    idx_90 = int(iterations * 0.90)
    idx_95 = int(iterations * 0.95)
    idx_995 = int(iterations * 0.995)
    
    mean_loss = float(torch.mean(annual_losses).item())
    var_90 = float(sorted_losses[idx_90].item())
    var_95 = float(sorted_losses[idx_95].item())
    var_99_5 = float(sorted_losses[idx_995].item())
    tvar_99_5 = float(torch.mean(sorted_losses[idx_995:]).item())
    
    # 清償資本要求 (SCR) 與加成校準 (Markup Loading)
    scr = max(0.0, var_99_5 - mean_loss)
    calibrated_markup = round(1.0 + (scr * 0.06) / max(1.0, mean_loss), 2)
    calibrated_markup = max(1.15, min(calibrated_markup, 3.0))
    
    return {
        "peril": peril_name,
        "iterations": iterations,
        "elapsed_ms": round(elapsed_ms, 2),
        "mean_annual_loss_usd": round(mean_loss, 2),
        "var_90_usd": round(var_90, 2),
        "var_95_usd": round(var_95, 2),
        "var_99_5_usd": round(var_99_5, 2),
        "tvar_99_5_usd": round(tvar_99_5, 2),
        "scr_usd": round(scr, 2),
        "calibrated_markup": calibrated_markup,
    }

In [ ]:
# 3. 執行四大新興風險之 1,000,000 次巨災壓力測試
results = {}
test_cases = [
    ("typhoon", 0.4516, 131250.0),    # 颱風侵襲嚴重事件 (NFA 統計)
    ("flood", 0.0968, 70312.5),       # 豪大雨致災事件
    ("earthquake", 0.0968, 384375.0),  # 地震強震事件
    ("cyber", 0.0800, 250000.0),      # 生成式 AI / 雲端中斷事件
]

print("=" * 65)
print("AMD ROCm GPU 1,000,000 Iterations Catastrophe Stress Test")
print("=" * 65)
for p, f, l in test_cases:
    res = simulate_catastrophe_rocm(p, f, l)
    results[p] = res
    print(f"[{p.upper():10s}] 99.5% VaR: USD {res['var_99_5_usd']:>12,.2f} | "
          f"99.5% TVaR: USD {res['tvar_99_5_usd']:>12,.2f} | "
          f"Calibrated Markup: {res['calibrated_markup']}x | "
          f"Latency: {res['elapsed_ms']}ms")
print("=" * 65)

In [ ]:
# 4. 匯出標準 JSON 數據至 ForeSure 專案
with open("amd_rocm_simulation_output.json", "w", encoding="utf-8") as fp:
    json.dump(results, fp, indent=2, ensure_ascii=False)
print("✅ 模擬完成，成果已封裝至 amd_rocm_simulation_output.json，可直接供決策桌與 Word 報告讀取。")